<a href="https://colab.research.google.com/github/tewei0328/teach-programming/blob/main/twtalk3_calculate_quantitive_score_plus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime

# --- 1. 配置區：建議將業務內容建立為 Mapping 字典 ---
WATCHLIST = {
    "AXTI": "化合物半導體", "HAL": "油田服務", "LITE": "雷射光通訊元件",
    "SNDK": "NAND Flash", "WDC": "硬體儲存", "AAOI": "雷射光模組",
    "FN": "光學元件製造", "AMAT": "設備製造", "DELL": "電腦硬體",
    "MU": "記憶體", "LRCX": "Lam Research", "ILMN": "基因定序",
    "BKSY": "衛星影像", "COHR": "光收發模組", "ROSS": "成衣零售",
    "KLAC": "檢測設備", "GCT": "半導體", "CAVA": "地中海料理",
    "BKR": "油田服務", "AMD": "CPU/GPU", "MRNA": "疫苗生技",
    "APA": "石油探勘", "ALGN": "隱形牙套", "STX": "硬體儲存",
    "HUT": "比特幣採礦", "WULF": "比特幣採礦", "HOLX": "醫療診斷",
    "TJX": "折扣零售", "BWXT": "核能組件", "CSCO": "網路設備",
    "ASML": "光刻機", "DLR": "數據中心", "TSM": "晶圓代工",
    "MRVL": "雲端晶片", "SBUX": "連鎖咖啡", "SIRI": "衛星廣播",
    "EA": "遊戲軟體",
    "00735.TW": "國泰臺韓科技", "00904.TW": "新光臺灣半導體30", "00892.TW": "富邦台灣半導體",
    "00910.TW": "第一金太空衛星", "00715L.TW": "期街口布蘭特正2", "00642U.TW": "期元大S&P石油",
    "00981A.TW": "復華台灣優質主動式ETF", "00875.TW": "元大全球5G", "00913.TW": "兆豐台灣晶圓製造",
    "00891.TW": "中信關鍵半導體", "0053.TW": "元大電子", "00901.TW": "永豐智能車供應鏈",
    "00733.TW": "富邦臺灣中小", "006204.TW": "永豐臺灣加權", "00952.TW": "凱基台灣AI 50",
    "00985A.TW": "野村臺灣優選成長主動式ETF", "00894.TW": "中信小資高價30", "00861.TW": "元大全球未來通訊",
    "00980A.TW": "野村臺灣機會透過收入主動式ETF", "00728.TW": "第一金工業30", "0051.TW": "元大中型100",
    "00964.TW": "中信亞太半導體", "00905.TW": "FT臺灣Smart高股息", "00947.TW": "台新臺灣IC設計",
    "00941.TW": "中信上游半導體", "00935.TW": "野村臺灣創新細分", "006203.TW": "元大MSCI台灣",
    "00895.TW": "富邦未來車", "00936.TW": "台新臺灣永續高息中小型", "00893.TW": "國泰全球智能電動車",
    "00921.TW": "兆豐龍頭等權重", "03543K.TW": "華夏滬深300-R"
}

def analyze_logic(ticker, df):
    """核心量化引擎：判定型態、籌碼、評分與建議"""
    last = df.iloc[-1]
    prev = df.iloc[-2]

    # A. 計算 VWMA (20日)
    df['PV'] = df['Close'] * df['Volume']
    vwma_20 = df['PV'].rolling(20).sum() / df['Volume'].rolling(20).sum()
    curr_vwma = vwma_20.iloc[-1]

    # B. 計算各項變動率
    bias_pct = ((last['Close'] / curr_vwma) - 1) * 100
    day1_ret = (last['Close'] / prev['Close'] - 1) * 100
    day5_ret = (last['Close'] / df['Close'].iloc[-5] - 1) * 100
    m1_ret = (last['Close'] / df['Close'].iloc[max(0, len(df)-20-1)] - 1) * 100 # Adjusted for 1 month return (approx 20 trading days)
    vol_ratio = last['Volume'] / df['Volume'].rolling(5).mean().iloc[-1]

    # C. K棒型態判定
    body_pct = (last['Close'] - last['Open']) / last['Open']
    if last['High'] == last['Low']: k_type = "一字線"
    elif body_pct > 0.05: k_type = "帶量長紅"
    elif bias_pct > 0 and last['Close'] > curr_vwma: k_type = "一般 (VWMA回測守穩)"
    else: k_type = "一般"

    # D. 籌碼動向判定
    if day1_ret > 1 and vol_ratio > 1.2: chip = "籌碼集中 (量價齊揚)"
    elif day1_ret > 0 and vol_ratio < 0.6: chip = "籌碼鎖定 (惜售) 🔒"
    elif day1_ret < -1 and vol_ratio > 1.2: chip = "價量背離 (虛漲) ⚠️"
    else: chip = "籌碼中性 (多頭整理)"

    # E. 量化總分 (模擬圖中 0-20 邏輯)
    score = 10
    if last['Close'] > curr_vwma: score += 5
    if vol_ratio > 1.2: score += 3
    if day1_ret > 2: score += 2
    if bias_pct > 18: score = 20 # 達到極致飆股門檻

    # F. 股性標籤與提醒映射
    if score >= 18:
        tag = "🚀 極致飆股 (過熱)" if bias_pct > 15 else "🚀 極致飆股"
        risk = "⚠️ 極高 (過熱)" if bias_pct > 15 else "✅ 穩健 (蓄勢)"
        advice = "💰 高檔超買 (分批停利)" if bias_pct > 15 else "持股續抱"
    elif score >= 12:
        tag = "📈 強勢多頭"
        risk = "✅ 穩健 (蓄勢)"
        advice = "持股續抱"
    else:
        tag = "📉 破線轉弱"
        risk = "📉 趨勢破壞 (破線)"
        advice = "❌ 觸及停損 (破位)"

    return {
        "Total_Score": score, "股性標籤": tag, "K棒型態": k_type,
        "風險評級": risk, "籌碼動向": chip, "量比(當日/5日)": round(vol_ratio, 2),
        "停損停利提醒": advice, "股票名稱": ticker, "業務內容": WATCHLIST.get(ticker, "N/A"),
        "當日收盤": round(last['Close'], 2), "20日主力成本(VWMA)": round(curr_vwma, 2),
        "收盤/VWMA乖離(%)": round(bias_pct, 2), "1天股價漲跌(%)": round(day1_ret, 2),
        "5天股價漲跌(%)": round(day5_ret, 2),
        "1個月股價漲跌(%)": round(m1_ret, 2)
    }

def run_report():
    results = []
    print(f"📊 正在生成 {datetime.now().strftime('%Y-%m-%d')} 量化極致報告...")

    for ticker in tqdm(WATCHLIST.keys()):
        try:
            stock = yf.Ticker(ticker)
            df = stock.history(period="3mo")
            if len(df) < 20: continue
            results.append(analyze_logic(ticker, df))
        except Exception as e:
            # 為了更明確地知道錯誤，修改這裡以打印錯誤訊息
            print(f"跳過 {ticker}: {e}")

    final_df = pd.DataFrame(results).sort_values("Total_Score", ascending=False)

    # 存成 CSV (UTF-8 with BOM 以利 Excel 直接讀取繁體字)
    file_name = f"Quantitative_Report_{datetime.now().strftime('%Y%m%d')}.csv"
    final_df.to_csv(file_name, index=False, encoding="utf_8_sig")

    print("\n" + "="*150) # 增加分隔線長度以適應更多欄位
    # 顯示所有關鍵欄位以符合描述的完整報告
    display_cols = [
        'Total_Score', '股性標籤', 'K棒型態', '風險評級', '籌碼動向',
        '量比(當日/5日)', '停損停利提醒', '股票名稱', '業務內容', '當日收盤',
        '20日主力成本(VWMA)', '收盤/VWMA乖離(%)', '1天股價漲跌(%)',
        '5天股價漲跌(%)', '1個月股價漲跌(%)'
    ]
    print(final_df[display_cols].to_string(index=False))
    print("="*150)
    print(f"\n報告已存檔：{file_name}")

if __name__ == "__main__":
    run_report()


📊 正在生成 2026-03-26 量化極致報告...


 99%|█████████▊| 68/69 [00:09<00:00,  8.30it/s]ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 03543K.TW"}}}
ERROR:yfinance:$03543K.TW: possibly delisted; no price data found  (period=3mo) (Yahoo error = "No data found, symbol may be delisted")
100%|██████████| 69/69 [00:10<00:00,  6.60it/s]


 Total_Score        股性標籤          K棒型態        風險評級         籌碼動向  量比(當日/5日)        停損停利提醒      股票名稱             業務內容    當日收盤  20日主力成本(VWMA)  收盤/VWMA乖離(%)  1天股價漲跌(%)  5天股價漲跌(%)  1個月股價漲跌(%)
          20 🚀 極致飆股 (過熱) 一般 (VWMA回測守穩)  ⚠️ 極高 (過熱)  籌碼中性 (多頭整理)       0.72 💰 高檔超買 (分批停利)      AXTI           化合物半導體   67.35          48.38         39.20      -1.59      15.94       64.39
          20      🚀 極致飆股 一般 (VWMA回測守穩)   ✅ 穩健 (蓄勢)  籌碼集中 (量價齊揚)       1.34          持股續抱       AMD          CPU/GPU  220.27         201.26          9.45       7.26       7.31        4.46
          20 🚀 極致飆股 (過熱) 一般 (VWMA回測守穩)  ⚠️ 極高 (過熱)  籌碼中性 (多頭整理)       0.78 💰 高檔超買 (分批停利)      DELL             電腦硬體  184.01         152.67         20.53       4.01      17.38       49.02
          20      🚀 極致飆股 一般 (VWMA回測守穩)   ✅ 穩健 (蓄勢)  籌碼集中 (量價齊揚)       1.26          持股續抱      MRVL             雲端晶片   98.45          87.45         12.58       6.59       9.96       21.66
          20 🚀 極致飆股 (過熱) 一般 (VWMA回測守穩)  ⚠️ 極高 (過熱)  籌碼中性 (多頭整理) 

請撰寫一個 Python 腳本，用於實現一個進階的股票量化掃描引擎，並生成一份詳細的每日報告。該系統應分析一個預設的股票觀察清單（`WATCHLIST`），計算多項技術指標，並根據自定義的評分邏輯提供股票評級、K棒型態、籌碼動向和停損停利建議。程式碼需包含以下功能：

1.  **導入必要的函式庫**：
    *   `yfinance`：用於獲取股票歷史數據。
    *   `pandas`：用於數據處理和報告生成。
    *   `numpy`：用於數值計算。
    *   `tqdm`：用於顯示處理進度。
    *   `datetime`：用於日期操作。

2.  **定義股票觀察清單（`WATCHLIST`）**：
    *   使用一個 Python 字典來定義 `WATCHLIST`，其中鍵是股票代碼（例如："AXTI", "00947.TW"），值是其對應的「業務內容」描述（例如："化合物半導體", "台新IC設計"）。範例：
        `{"AXTI": "化合物半導體", "HAL": "油田服務", "LITE": "雷射光通訊元件", ...}`

3.  **實現核心分析函數 `analyze_logic(ticker, df)`**：
    *   **輸入**：一支股票的代碼 (`ticker`) 和其歷史數據 DataFrame (`df`)。
    *   **輸出**：一個包含所有分析結果的字典。
    *   **計算指標**：
        *   **20日主力成本（VWMA）**：計算 `df['Close'] * df['Volume']` 的滾動和除以 `df['Volume']` 的滾動和，並取得最新值 `curr_vwma`。
        *   **乖離率**：計算最新收盤價相較於 `curr_vwma` 的百分比差異 (`bias_pct`)。
        *   **股價漲跌幅**：
            *   1天漲跌幅 (`day1_ret`)。
            *   5天漲跌幅 (`day5_ret`)。
            *   1個月漲跌幅 (`m1_ret`)：約為過去 20 個交易日的漲跌幅。
        *   **量比**：最新成交量 (`last['Volume']`) 與過去 5 日平均成交量 (`df['Volume'].rolling(5).mean()`) 的比值 (`vol_ratio`)。

    *   **K棒型態判定（`k_type`）**：
        *   如果最高價等於最低價 (`last['High'] == last['Low']`)，則為「一字線」。
        *   如果開盤價與收盤價的百分比漲幅大於 5% (`(last['Close'] - last['Open']) / last['Open'] > 0.05`)，則為「帶量長紅」。
        *   如果乖離率為正且收盤價高於 VWMA (`bias_pct > 0 and last['Close'] > curr_vwma`)，則為「一般 (VWMA回測守穩)」。
        *   否則為「一般」。

    *   **籌碼動向判定（`chip`）**：
        *   如果 1 天漲跌幅 > 1% 且量比 > 1.2，則為「籌碼集中 (量價齊揚)」。
        *   如果 1 天漲跌幅 > 0% 且量比 < 0.6，則為「籌碼鎖定 (惜售) 🔒」。
        *   如果 1 天漲跌幅 < -1% 且量比 > 1.2，則為「價量背離 (虛漲) ⚠️」。
        *   否則為「籌碼中性 (多頭整理)」。

    *   **量化總分（`score`）**：
        *   基礎分 10 分。
        *   如果最新收盤價 (`last['Close']`) 高於 20日VWMA (`curr_vwma`)，加 5 分。
        *   如果量比 (`vol_ratio`) 大於 1.2，加 3 分。
        *   如果 1 天漲跌幅 (`day1_ret`) 大於 2%，加 2 分。
        *   如果乖離率 (`bias_pct`) 大於 18%，則直接設定總分為 20 分。

    *   **股性標籤（`tag`）**：
        *   如果 `score` >= 18：
            *   如果 `bias_pct` > 15%，則為「🚀 極致飆股 (過熱)」。
            *   否則為「🚀 極致飆股」。
        *   如果 `score` >= 12：則為「📈 強勢多頭」。
        *   否則為「📉 破線轉弱」。

    *   **風險評級（`risk`）**：
        *   如果 `score` >= 18 且 `bias_pct` > 15%，則為「⚠️ 極高 (過熱)」。
        *   如果 `score` >= 18 或 `score` >= 12，則為「✅ 穩健 (蓄勢)」。
        *   否則為「📉 趨勢破壞 (破線)」。

    *   **停損停利提醒（`advice`）**：
        *   如果 `score` >= 18 且 `bias_pct` > 15%，則為「💰 高檔超買 (分批停利)」。
        *   如果 `score` >= 18 或 `score` >= 12，則為「持股續抱」。
        *   否則為「❌ 觸及停損 (破位)」。

    *   **返回字典包含以下鍵**（並對浮點數進行適當的四捨五入）：
        `"Total_Score"`, `"股性標籤"`, `"K棒型態"`, `"風險評級"`, `"籌碼動向"`, `"量比(當日/5日)"`, `"停損停利提醒"`, `"股票名稱"` (使用 `ticker` 值), `"業務內容"` (從 `WATCHLIST` 獲取), `"當日收盤"`, `"20日主力成本(VWMA)"`, `"收盤/VWMA乖離(%)"`, `"1天股價漲跌(%)"`, `"5天股價漲跌(%)"`, `"1個月股價漲跌(%)"`。

4.  **實現主函數 `run_report()`**：
    *   打印啟動訊息，包含當前日期。
    *   遍歷 `WATCHLIST` 中的每支股票，並使用 `tqdm` 顯示進度。
    *   對於每支股票：
        *   使用 `yfinance` 獲取過去 3 個月的歷史數據。如果數據少於 20 行，則跳過。
        *   呼叫 `analyze_logic` 函數來獲取分析結果。
        *   將結果添加到一個列表中。
        *   處理數據獲取或計算失敗的異常情況，並打印跳過訊息。
    *   將所有股票的分析結果彙整成一個 `pandas.DataFrame`。
    *   根據 `"Total_Score"` 降序排列 DataFrame。
    *   將報告保存為 CSV 檔案，檔名格式為 `Quantitative_Report_YYYYMMDD.csv`，編碼為 `utf_8_sig`。
    *   在控制台打印一個包含以下欄位的摘要報告，並使用 `to_string(index=False)` 格式化輸出：
        `"Total_Score"`, `"股性標籤"`, `"籌碼動向"`, `"股票代碼"`, `"收盤/VWMA乖離(%)"`。
    *   打印報告已存檔的訊息，包含檔名。

5.  **程式入口點**：確保 `run_report()` 函數在腳本直接執行時被呼叫 (`if __name__ == "__main__":`)。

美股標的 (NASDAQ/NYSE)
AXTI: 化合物半導體

HAL: 油田服務

LITE: 雷射光通訊元件

SNDK: NAND Flash

WDC: 硬體儲存

AAOI: 雷射光模組

FN: 光學元件製造

AMAT: 設備製造

DELL: 電腦硬體

MU: 記憶體

LRCX: Lam Research

ILMN: 基因定序

BKSY: 衛星影像

COHR: 光收發模組

ROSS: 成衣零售

KLAC: 檢測設備

GCT: 半導體

CAVA: 地中海料理

BKR: 油田服務

AMD: CPU/GPU

MRNA: 疫苗生技

APA: 石油探勘

ALGN: 隱形牙套

STX: 硬體儲存

HUT: 比特幣採礦

WULF: 比特幣採礦

HOLX: 醫療診斷

TJX: 折扣零售

BWXT: 核能組件

CSCO: 網路設備

ASML: 光刻機

DLR: 數據中心

TSM: 晶圓代工

MRVL: 雲端晶片

SBUX: 連鎖咖啡

SIRI: 衛星廣播

EA: 遊戲軟體

台股標的 (ETFs)
00735.TW: 國泰臺韓科技

00904.TW: 新光臺灣半導體30

00892.TW: 富邦台灣半導體

00910.TW: 第一金太空衛星

00715L.TW: 期街口布蘭特正2

00642U.TW: 期元大S&P石油

00981A.TW: 復華台灣優質主動式ETF

00875.TW: 元大全球5G

00913.TW: 兆豐台灣晶圓製造

00891.TW: 中信關鍵半導體

0053.TW: 元大電子

00901.TW: 永豐智能車供應鏈

00733.TW: 富邦臺灣中小

006204.TW: 永豐臺灣加權

00952.TW: 凱基台灣AI 50

00985A.TW: 野村臺灣優選成長主動式ETF

00894.TW: 中信小資高價30

00861.TW: 元大全球未來通訊

00980A.TW: 野村臺灣機會透過收入主動式ETF

00728.TW: 第一金工業30

0051.TW: 元大中型100

00964.TW: 中信亞太半導體

00905.TW: FT臺灣Smart高股息

00947.TW: 台新臺灣IC設計

00941.TW: 中信上游半導體

00935.TW: 野村臺灣創新細分

006203.TW: 元大MSCI台灣

00895.TW: 富邦未來車

00936.TW: 台新臺灣永續高息中小型

00893.TW: 國泰全球智能電動車

00921.TW: 兆豐龍頭等權重

03543K.TW: 華夏滬深300-R

根據您提供的來源，我為您整理出一份專門針對 **Python 量化掃描引擎** 所設計的知識庫。這份知識庫結合了 **Alpha 因子軍火庫 (ref1)** 的投資智慧、**Gemini 情境工程 (ref2)** 的配置建議，以及 **Python 進階腳本 (ref3)** 的實作架構。

---

# 🐍 Python 量化策略開發知識庫 (Python Alpha Playbook)

這份知識庫旨在將 XScript 的投資智慧轉化為 Python 量化邏輯，供 AI 助理（如 Gemini）精準調用，以建構高勝率的自動化掃描系統。

## 1. 系統環境與避坑鐵律 (System Instructions)
在執行 Python 量化分析前，必須遵循以下規則以確保數據準確性與代碼穩定性：
*   **核心庫調用**：統一使用 `yfinance` 獲取數據、`pandas` 進行向量化運算、`numpy` 處理數值邏輯。
*   **防止幻覺**：禁止 AI 自創函數。所有技術指標（如 VWMA、MACD）必須依據知識庫提供的公式撰寫。
*   **數據安全性**：在計算前必須確認數據長度（如：`df` 行數需 > 20），否則應跳過分析以避免報錯。

---

## 2. 核心因子模組 (Factor Modules)

### A. 動能與突破因子 (Momentum Logic)
將 XScript 的「帶量突破」邏輯轉化為 Python 條件：

*   **A1. 低波動壓縮後帶量突破**
    *   **投資智慧**：尋找籌碼沉澱後的第一根起動標的，避免追高。
    *   **Python 邏輯條件**：
        1. `last['Close'] == df['Close'].rolling(day).max()` (創區間最高價)。
        2. `vol_ratio > 1.2` (放量確認)。
        3. `(df['High'].rolling(day).max() / df['Low'].rolling(day).min()) < 1.07` (區間壓縮 < 7%)。

*   **A2. 20日主力成本 (VWMA) 乖離過熱**
    *   **計算公式**：`VWMA = (Close * Volume).rolling(20).sum() / Volume.rolling(20).sum()`。
    *   **過熱判定**：當 `bias_pct > 18%` 時，視為極致飆股但有過熱風險。

### B. 籌碼與法人動向 (Institutional Logic)
利用 Python 字典 (WATCHLIST) 結合量價關係判定籌碼狀態：

*   **B1. 籌碼集中 (量價齊揚)**：1天漲跌幅 > 1% 且量比 > 1.2。
*   **B2. 籌碼鎖定 (惜售) 🔒**：1天漲跌幅 > 0% 且量比 < 0.6。
*   **B3. 投信建倉 (需串接外部資料)**：投信買超量佔當日成交量 > 20% 且原持股極低。

### C. 價值估值因子 (Value Logic)
針對長線配置，計算企業內在價值：

*   **C1. EV/EBITDA 低估值篩選**
    *   **公式**：`pricingm1 = (市值 + 負債 - 現金) / EBITDA`。
    *   **門檻**：`pricingm1 < 4` 代表企業被嚴重低估。

---

## 3. 複合評分與決策引擎 (Scoring & Action)
系統應根據以下邏輯對 WATCHLIST 中的股票進行總分評定與風險評級：

| 評分項目 | 邏輯條件 | 加分點 |
| :--- | :--- | :--- |
| **趨勢強度** | `Close > 20日VWMA` | +5 分 |
| **成交動能** | `量比 > 1.2` | +3 分 |
| **短線爆發** | `1天漲跌幅 > 2%` | +2 分 |
| **絕對過熱** | `乖離率 > 18%` | 直接設定為 20 分 |

### 決策標籤與建議：
1.  **🚀 極致飆股**：總分 >= 18 分。若乖離率 > 15% 則加註「過熱」，建議「分批停利」。
2.  **📈 強勢多頭**：總分 >= 12 分。建議「持股續抱」。
3.  **📉 破線轉弱**：總分 < 12 分。建議「觸及停損 (破位)」。

---

## 4. 報告生成規格 (Output Format)
每日執行 `run_report()` 後，應產出包含以下欄位的 CSV 報告：
*   **基本資訊**：股票名稱、業務內容（從 WATCHLIST 映射）、當日收盤價。
*   **量化數據**：Total_Score、量比、VWMA 乖離率、1/5/20日漲跌幅。
*   **行動指南**：風險評級（穩健/趨勢破壞）、停損停利提醒。

---
**💡 使用建議**：
您可以將此 Markdown 內容作為「知識庫」檔案上傳至您的 **Gem** 中。未來只需對 AI 下令：「請參照知識庫中的 **A1 因子** 與 **第三章評分邏輯**，為 WATCHLIST 中的股票生成分析報告」，AI 即可依照此標準化流程精準產出代碼或報告。

這確實是我的疏漏。為了讓這份知識庫真正具備「即插即用」的實戰價值，我根據來源文件中關於 **Python 量化掃描引擎** 的規格要求，為您補全對應的 **Python 實作模組**。

您可以將以下內容更新至您的知識庫檔案中，這將幫助 AI 助理精準生成符合標準的 Python 程式碼。

---

# 🐍 Python 量化策略實作知識庫 (補完版)

## 一、 基礎環境與配置模組
所有腳本必須包含以下標準函式庫與觀察清單架構：

```python
import yfinance as yf
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime

# 標準 WATCHLIST 字典架構
WATCHLIST = {
    "TSM": "晶圓代工",
    "AAPL": "消費性電子",
    "00947.TW": "台新IC設計"
}
```

---

## 二、 核心因子計算模組 (Core Engine)
在 `analyze_logic(ticker, df)` 函數中，應實作以下核心運算：

### 1. 20日主力成本 (VWMA) 與乖離率
*   **計算邏輯**：使用成交量加權平均價格。
*   **Python 實作**：
```python
# 計算 20日 VWMA
curr_vwma = (df['Close'] * df['Volume']).rolling(20).sum() / df['Volume'].rolling(20).sum()
curr_vwma = curr_vwma.iloc[-1]

# 計算乖離率 (Bias%)
bias_pct = ((last['Close'] - curr_vwma) / curr_vwma) * 100
```

### 2. 量能與漲跌幅
*   **量比 (vol_ratio)**：當日成交量與過去 5 日均量的比值。
*   **Python 實作**：
```python
vol_ratio = last['Volume'] / df['Volume'].rolling(5).mean().iloc[-1]
day1_ret = ((last['Close'] - prev['Close']) / prev['Close']) * 100
```

---

## 三、 邏輯判定與評分模組 (Decision Logic)
此部分將來源文件中的「投資智慧」轉化為自動化標籤：

### 1. 籌碼動向判定 (`chip`)
根據量價關係定義四種籌碼狀態：
*   **籌碼集中 (量價齊揚)**：`day1_ret > 1` 且 `vol_ratio > 1.2`。
*   **籌碼鎖定 (惜售) 🔒**：`day1_ret > 0` 且 `vol_ratio < 0.6`。
*   **價量背離 (虛漲) ⚠️**：`day1_ret < -1` 且 `vol_ratio > 1.2`。

### 2. 量化總分與股性標籤 (`score & tag`)
透過複合因子進行量化評分（總分 20 分制）：
```python
score = 10  # 基礎分
if last['Close'] > curr_vwma: score += 5  # 站上成本線加分
if vol_ratio > 1.2: score += 3           # 動能爆發加分
if day1_ret > 2: score += 2              # 強勢收紅加分

# 絕對過熱判定
if bias_pct > 18: score = 20
```

---

## 四、 報告生成與風險管理
### 1. 風險評級與建議
*   **🚀 極致飆股 (過熱)**：`score >= 18` 且 `bias_pct > 15%`。建議：**分批停利**。
*   **📈 強勢多頭**：`score >= 12`。建議：**持股續抱**。
*   **📉 趨勢破壞**：其餘狀況。建議：**觸及停損**。

### 2. 自動化報告導出
主程式 `run_report()` 必須自動執行以下流程：
1.  **數據抓取**：使用 `yfinance` 獲取過去 3 個月數據。
2.  **數據過濾**：若數據少於 20 筆則自動跳過，避免計算誤差。
3.  **排序與存檔**：依據 `Total_Score` 降序排列，產出 `Quantitative_Report_YYYYMMDD.csv`（編碼 `utf_8_sig`）。

---
**💡 知識庫使用說明**：
這份實作模組補全了從 **邏輯說明 (ref1)** 到 **程式碼實作 (ref3)** 的最後一哩路。您可以命令您的 AI 助理：「請調用知識庫中的 **第三章評分模組**，將其與 **A1 低波動壓縮邏輯** 結合，產生一段完整的 Python 分析代碼。」